# 네이버 블로그/카페글 검색 API 크롤링

네이버 검색 API로 블로그와 카페글 후보를 수집합니다. 이 노트북은 **완화 조건 전용**입니다.

- 검색 API는 본문 전체가 아니라 `title`과 `description`만 제공합니다.
- 각 키워드 조합 4개 중 `MIN_KEYWORD_MATCH_COUNT`개 이상이 `title + description`에 포함된 글만 남깁니다.
- 5개 큰 주제는 서로 섞이지 않도록 주제별로 따로 수집하고 저장합니다.
- 저장 파일은 `crawl_data_비염.csv`, `crawl_data_탈모.csv`처럼 주제별 1개씩 생성됩니다.
- 같은 주제 안에서 동일 URL이 여러 키워드 조합에 잡히면 URL 기준으로 1행만 남기고, 검색 키워드 정보는 한 칸에 합쳐 보존합니다.

In [ ]:
import os
import re
import time
from datetime import datetime
from html import unescape
from itertools import combinations
from pathlib import Path

import pandas as pd
import requests

BASE_DIR = Path(os.getenv("DATA_JOUR_BASE_DIR", ".")).expanduser().resolve()

try:
    from dotenv import load_dotenv
    load_dotenv(BASE_DIR / ".env")
except ImportError:
    print("python-dotenv가 설치되어 있지 않으면 .env 자동 로드는 생략됩니다.")
except Exception as e:
    print(f".env 자동 로드 중 오류가 발생했습니다: {e}")

## 1. 키워드 그룹과 수집 옵션 설정

`KEYWORD_GROUPS`의 각 key가 저장 파일명에 들어가는 핵심키워드입니다. 예를 들어 `"비염"` 그룹은 `crawl_data_비염.csv`로 저장됩니다.

In [ ]:
KEYWORD_GROUPS = {
    "비염": [
        ["비염", "비염스프레이", "프로폴리스", "후기"],
        ["비염", "비염스프레이", "프로폴리스", "효과"],
        ["비염", "비염스프레이", "프로폴리스", "추천"],
        ["비염", "비염 영양제", "프로폴리스", "후기"],
        ["비염", "비염 영양제", "프로폴리스", "효과"],
        ["비염", "비염 영양제", "프로폴리스", "추천"],
    ],
    "탈모": [
        ["탈모", "탈모약", "미녹시딜", "후기"],
        ["탈모", "탈모약", "미녹시딜", "효과"],
        ["탈모", "탈모약", "미녹시딜", "추천"],
        ["탈모", "탈모앰플", "미녹시딜", "후기"],
        ["탈모", "탈모앰플", "미녹시딜", "효과"],
        ["탈모", "탈모앰플", "미녹시딜", "추천"],
    ],
    "피부미용": [
        ["피부미용", "피부영양제", "콜라겐", "후기"],
        ["피부미용", "피부영양제", "콜라겐", "효과"],
        ["피부미용", "피부영양제", "콜라겐", "추천"],
        ["피부미용", "이너뷰티", "콜라겐", "후기"],
        ["피부미용", "이너뷰티", "콜라겐", "효과"],
        ["피부미용", "이너뷰티", "콜라겐", "추천"],
    ],
    "키성장": [
        ["키성장", "키영양제", "황기추출물", "후기"],
        ["키성장", "키영양제", "황기추출물", "효과"],
        ["키성장", "키영양제", "황기추출물", "추천"],
        ["키성장", "키크는영양제", "황기추출물", "후기"],
        ["키성장", "키크는영양제", "황기추출물", "효과"],
        ["키성장", "키크는영양제", "황기추출물", "추천"],
    ],
    "다이어트": [
        ["다이어트", "다이어트보조제", "가르시니아", "후기"],
        ["다이어트", "다이어트보조제", "가르시니아", "효과"],
        ["다이어트", "다이어트보조제", "가르시니아", "추천"],
        ["다이어트", "다이어트유산균", "가르시니아", "후기"],
        ["다이어트", "다이어트유산균", "가르시니아", "효과"],
        ["다이어트", "다이어트유산균", "가르시니아", "추천"],
    ],
}

# 4개 키워드 중 몇 개 이상 title + description에 포함되어야 최종 수집할지 설정합니다.
MIN_KEYWORD_MATCH_COUNT = 3

# source별 최대 수집 시도 건수입니다. 네이버 검색 API start는 최대 1000까지 사용할 수 있습니다.
MAX_ITEMS_PER_SOURCE = 300

# 한 번 요청할 때 가져올 결과 수입니다. 네이버 검색 API의 display 최댓값은 100입니다.
DISPLAY = 100

# 정렬 방식: sim=정확도순, date=날짜순
SORT = "sim"

# 요청 사이 대기 시간입니다. 너무 빠른 반복 요청을 피하기 위한 값입니다.
RATE_LIMIT_SEC = 0.2

# 테스트 크롤링 기본값입니다. 처음에는 한 주제만 작게 테스트한 뒤 전체 수집을 실행하세요.
TEST_CATEGORY_NAMES = ["비염"]
TEST_MAX_ITEMS_PER_SOURCE = 10
TEST_DISPLAY = 10

# 전체 수집할 주제입니다. 5개 파일을 모두 만들려면 기본값 그대로 둡니다.
TARGET_CATEGORY_NAMES = list(KEYWORD_GROUPS.keys())

## 2. API 호출, 필터링, 저장 함수

In [ ]:
def clean_html(text):
    """네이버 검색 API 응답에 포함된 HTML 태그와 엔티티를 제거합니다."""
    if text is None:
        return ""
    text = re.sub(r"<[^>]+>", "", str(text))
    return unescape(text).strip()


def safe_filename(value):
    value = str(value).strip()
    value = re.sub(r'[\\/:*?"<>|]+', "_", value)
    value = re.sub(r"\s+", "_", value)
    return value


def get_category_output_path(core_keyword):
    return BASE_DIR / f"crawl_data_{safe_filename(core_keyword)}.csv"


def get_naver_headers():
    client_id = os.getenv("NAVER_CLIENT_ID", "").strip().strip("'\"")
    client_secret = os.getenv("NAVER_CLIENT_SECRET", "").strip().strip("'\"")

    if not client_id or not client_secret:
        raise ValueError(
            "NAVER_CLIENT_ID 또는 NAVER_CLIENT_SECRET이 없습니다. "
            ".env 파일이나 환경변수에 네이버 API 키를 설정하세요."
        )

    return {
        "X-Naver-Client-Id": client_id,
        "X-Naver-Client-Secret": client_secret,
    }


def validate_keywords(keywords):
    keywords = [str(keyword).strip() for keyword in keywords if str(keyword).strip()]
    if len(keywords) != 4:
        raise ValueError("각 키워드 조합에는 비어 있지 않은 키워드 4개를 입력해야 합니다.")
    return keywords


def validate_keyword_groups(keyword_groups):
    if not isinstance(keyword_groups, dict) or not keyword_groups:
        raise ValueError("KEYWORD_GROUPS는 비어 있지 않은 dict여야 합니다.")

    validated = {}
    for core_keyword, keyword_sets in keyword_groups.items():
        core_keyword = str(core_keyword).strip()
        if not core_keyword:
            raise ValueError("핵심키워드 이름은 비어 있을 수 없습니다.")
        validated[core_keyword] = [validate_keywords(keywords) for keywords in keyword_sets]
        if not validated[core_keyword]:
            raise ValueError(f"{core_keyword} 그룹에 키워드 조합이 없습니다.")
    return validated


def normalize_item(item, source):
    title = clean_html(item.get("title", ""))
    description = clean_html(item.get("description", ""))

    row = {
        "source": source,
        "title": title,
        "description": description,
        "link": item.get("link", ""),
        "postdate": item.get("postdate", ""),
        "collected_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    }

    if source == "blog":
        row.update({
            "site_name": clean_html(item.get("bloggername", "")),
            "site_url": item.get("bloggerlink", ""),
        })
    elif source == "cafearticle":
        row.update({
            "site_name": clean_html(item.get("cafename", "")),
            "site_url": item.get("cafeurl", ""),
        })
    else:
        row.update({"site_name": "", "site_url": ""})

    return row


def search_naver(source, query, headers, display=100, max_items=300, sort="date", rate_limit_sec=0.2):
    """네이버 검색 API에서 블로그 또는 카페글 검색 결과를 페이지네이션으로 수집합니다."""
    if source not in {"blog", "cafearticle"}:
        raise ValueError("source는 'blog' 또는 'cafearticle'만 사용할 수 있습니다.")

    display = min(int(display), 100)
    max_items = min(int(max_items), 1000)
    url = f"https://openapi.naver.com/v1/search/{source}.json"

    all_items = []
    start = 1

    while start <= max_items:
        params = {
            "query": query,
            "display": display,
            "start": start,
            "sort": sort,
        }

        response = requests.get(url, headers=headers, params=params, timeout=10)

        if response.status_code == 429:
            print(f"[{source}] 429 Too Many Requests: 5초 대기 후 재시도")
            time.sleep(5)
            continue

        if response.status_code != 200:
            print(f"[{source}] Error {response.status_code}: {response.text}")
            break

        data = response.json()
        items = data.get("items", [])
        if not items:
            break

        all_items.extend(items)
        print(f"[{source}] {start}~{start + len(items) - 1} 수집 / 누적 {len(all_items)}건 / 전체 {data.get('total', 0)}건")

        start += display
        if start > min(data.get("total", 0), max_items):
            break

        time.sleep(rate_limit_sec)

    return all_items


def count_matched_keywords(title, description, keywords):
    search_text = f"{clean_html(title)} {clean_html(description)}".lower()
    matched_keywords = [keyword for keyword in keywords if keyword.lower() in search_text]
    return len(matched_keywords), matched_keywords


def contains_min_keywords(title, description, keywords, min_match_count=3):
    matched_count, _ = count_matched_keywords(title, description, keywords)
    return matched_count >= min_match_count


def make_keyword_combination_queries(keywords, min_match_count=3):
    return [" ".join(combo) for combo in combinations(keywords, min_match_count)]


def add_keyword_metadata(row, core_keyword, keyword_set_id, keywords, full_keyword_query, api_query):
    matched_count, matched_keywords = count_matched_keywords(row.get("title", ""), row.get("description", ""), keywords)
    row["core_keyword"] = core_keyword
    row["keyword_set_id"] = keyword_set_id
    row["search_keywords"] = ", ".join(keywords)
    row["search_query"] = full_keyword_query
    row["api_query"] = api_query
    row["matched_keyword_count"] = matched_count
    row["matched_keywords"] = ", ".join(matched_keywords)
    return row


def crawl_category_min_keywords(
    core_keyword,
    keyword_sets,
    min_match_count=3,
    display=100,
    max_items_per_source=300,
    sort="date",
    rate_limit_sec=0.2,
):
    keyword_sets = [validate_keywords(keywords) for keywords in keyword_sets]
    if min_match_count < 1 or min_match_count > 4:
        raise ValueError("min_match_count는 1 이상 4 이하로 설정해야 합니다.")

    headers = get_naver_headers()
    rows = []

    for keyword_set_id, keywords in enumerate(keyword_sets, start=1):
        full_keyword_query = " ".join(keywords)
        api_queries = make_keyword_combination_queries(keywords, min_match_count)

        for source in ["blog", "cafearticle"]:
            for api_query in api_queries:
                print(f"[{core_keyword}][{source}] 키워드 세트 {keyword_set_id} 검색어: {api_query}")
                items = search_naver(
                    source=source,
                    query=api_query,
                    headers=headers,
                    display=display,
                    max_items=max_items_per_source,
                    sort=sort,
                    rate_limit_sec=rate_limit_sec,
                )

                for item in items:
                    if contains_min_keywords(
                        item.get("title", ""),
                        item.get("description", ""),
                        keywords,
                        min_match_count=min_match_count,
                    ):
                        row = normalize_item(item, source)
                        rows.append(add_keyword_metadata(row, core_keyword, keyword_set_id, keywords, full_keyword_query, api_query))

    df = pd.DataFrame(rows)
    if not df.empty:
        df = df.drop_duplicates(subset=["core_keyword", "keyword_set_id", "source", "link", "api_query"]).reset_index(drop=True)
    return df


def join_unique_values(values):
    seen = []
    for value in values:
        value = str(value).strip()
        if not value or value in seen:
            continue
        seen.append(value)
    return " | ".join(seen)


def deduplicate_category_posts_by_url(df):
    if df.empty:
        return df

    if "core_keyword" not in df.columns or "link" not in df.columns:
        raise ValueError("중복 제거에는 core_keyword와 link 컬럼이 필요합니다.")

    core_keywords = sorted(df["core_keyword"].dropna().astype(str).unique())
    if len(core_keywords) != 1:
        raise ValueError(f"한 CSV에는 하나의 core_keyword만 있어야 합니다. 현재 값: {core_keywords}")

    df = df[df["link"].astype(str).str.strip() != ""].copy()
    merge_columns = [
        "keyword_set_id",
        "search_keywords",
        "search_query",
        "api_query",
        "matched_keywords",
    ]

    agg_map = {col: "first" for col in df.columns if col != "link"}
    for col in merge_columns:
        if col in df.columns:
            agg_map[col] = join_unique_values

    df_dedup = (
        df
        .groupby("link", as_index=False, sort=False)
        .agg(agg_map)
        .reset_index(drop=True)
    )
    return df_dedup.reindex(columns=df.columns)


def validate_single_category(df, core_keyword):
    if df.empty:
        return
    values = sorted(df["core_keyword"].dropna().astype(str).unique())
    if values != [core_keyword]:
        raise ValueError(f"{core_keyword} 저장 데이터에 다른 주제가 섞였습니다. 현재 값: {values}")


def crawl_and_save_categories(
    keyword_groups,
    category_names=None,
    min_match_count=3,
    display=100,
    max_items_per_source=300,
    sort="date",
    rate_limit_sec=0.2,
    save=True,
):
    keyword_groups = validate_keyword_groups(keyword_groups)
    category_names = category_names or list(keyword_groups.keys())

    results = {}
    for core_keyword in category_names:
        if core_keyword not in keyword_groups:
            raise ValueError(f"KEYWORD_GROUPS에 없는 주제입니다: {core_keyword}")

        print(f"\n===== {core_keyword} 수집 시작 =====")
        df_raw = crawl_category_min_keywords(
            core_keyword=core_keyword,
            keyword_sets=keyword_groups[core_keyword],
            min_match_count=min_match_count,
            display=display,
            max_items_per_source=max_items_per_source,
            sort=sort,
            rate_limit_sec=rate_limit_sec,
        )
        df_dedup = deduplicate_category_posts_by_url(df_raw)
        validate_single_category(df_dedup, core_keyword)

        output_csv = get_category_output_path(core_keyword)
        if save:
            df_dedup.to_csv(output_csv, index=False, encoding="utf-8-sig")

        print(f"[{core_keyword}] API 필터링 후 행 수: {len(df_raw):,}")
        print(f"[{core_keyword}] URL 중복 제거 후 행 수: {len(df_dedup):,}")
        if save:
            print(f"[{core_keyword}] 저장 완료: {output_csv}")
        else:
            print(f"[{core_keyword}] 테스트 모드: 파일 저장 안 함")

        results[core_keyword] = df_dedup

    return results

## 3. 테스트 크롤링

처음에는 `TEST_CATEGORY_NAMES`를 한 주제만 두고 실행하세요. 결과가 정상적으로 나오면 전체 수집 단계로 넘어갑니다.

In [ ]:
test_results = crawl_and_save_categories(
    keyword_groups=KEYWORD_GROUPS,
    category_names=TEST_CATEGORY_NAMES,
    min_match_count=MIN_KEYWORD_MATCH_COUNT,
    display=TEST_DISPLAY,
    max_items_per_source=TEST_MAX_ITEMS_PER_SOURCE,
    sort=SORT,
    rate_limit_sec=RATE_LIMIT_SEC,
    save=False,
)

for core_keyword, df_test in test_results.items():
    print(f"{core_keyword} 테스트 결과: {len(df_test):,}건")
    display(df_test.head(10))

## 4. 전체 수집 및 주제별 CSV 저장

아래 셀을 실행하면 `TARGET_CATEGORY_NAMES`에 들어 있는 5개 주제에 대해 각각 별도 CSV를 저장합니다.

저장 파일:
- `crawl_data_비염.csv`
- `crawl_data_탈모.csv`
- `crawl_data_피부미용.csv`
- `crawl_data_키성장.csv`
- `crawl_data_다이어트.csv`

In [ ]:
category_results = crawl_and_save_categories(
    keyword_groups=KEYWORD_GROUPS,
    category_names=TARGET_CATEGORY_NAMES,
    min_match_count=MIN_KEYWORD_MATCH_COUNT,
    display=DISPLAY,
    max_items_per_source=MAX_ITEMS_PER_SOURCE,
    sort=SORT,
    rate_limit_sec=RATE_LIMIT_SEC,
)

summary_rows = []
for core_keyword, df_category in category_results.items():
    output_csv = get_category_output_path(core_keyword)
    summary_rows.append({
        "core_keyword": core_keyword,
        "rows": len(df_category),
        "unique_urls": df_category["link"].nunique() if "link" in df_category.columns and not df_category.empty else 0,
        "output_csv": str(output_csv),
    })

summary_df = pd.DataFrame(summary_rows)
summary_df

## 5. 저장 파일 점검

각 CSV에 다른 주제 데이터가 섞이지 않았는지 확인합니다.

In [ ]:
check_rows = []
for core_keyword in KEYWORD_GROUPS.keys():
    path = get_category_output_path(core_keyword)
    if not path.exists():
        check_rows.append({
            "core_keyword": core_keyword,
            "file_exists": False,
            "rows": 0,
            "unique_core_keywords": "",
            "duplicate_url_rows": "",
            "path": str(path),
        })
        continue

    df_check = pd.read_csv(path, encoding="utf-8-sig", dtype=str).fillna("")
    unique_core_keywords = sorted(df_check.get("core_keyword", pd.Series(dtype=str)).astype(str).unique())
    duplicate_url_rows = int(df_check.duplicated(subset=["link"], keep=False).sum()) if "link" in df_check.columns else None
    check_rows.append({
        "core_keyword": core_keyword,
        "file_exists": True,
        "rows": len(df_check),
        "unique_core_keywords": ", ".join(unique_core_keywords),
        "duplicate_url_rows": duplicate_url_rows,
        "path": str(path),
    })

pd.DataFrame(check_rows)